In [ ]:
import json
from difflib import SequenceMatcher
from tqdm import tqdm

# =======================
# CONFIG
# =======================

DEBUG = True        # 是否印出除錯資訊
VERIFY_SPAN = True  # 是否驗證 hallucination span 對齊

In [ ]:
# =======================
# PROMPT PARSING
# =======================

def build_prompt_user_only(entry, debug=False):
    parts = []

    for p in entry.get("prompt", {}).get("rag_prompt", []):
        if p.get("role") == "user":
            content = p.get("content")
            if content:
                parts.append(content.strip())

    if not parts:
        if debug:
            print("No user prompt found")
        return None

    if debug and len(parts) > 2:
        print("User prompt has >2 segments")

    return "\n\n".join(parts)

In [ ]:
# =======================
# HALLUCINATION LABEL
# =======================
def get_label(entry):
    for s in entry.get("sentence_data", []):
        if s.get("target") == "hallucinated":
            return 1
    return 0

In [ ]:
# =======================
# HALLUCINATION SPAN
# =======================

def reverse_hallucination_span(entry):
    response = entry.get("llm_response")
    if not response:
        return None

    prev = ""

    for s in entry.get("sentence_data", []):
        if s.get("target") == "hallucinated":
            cur = s.get("cum_sentence", "")

            delta = cur[len(prev):].strip()
            if not delta:
                return None

            start = response.find(delta)
            if start == -1:
                return None

            return {
                "start": start,
                "end": start + len(delta)
            }

        prev = s.get("cum_sentence", "")

    return None

In [ ]:
# =======================
# SPAN VERIFICATION
# =======================

def verify_hallucination_span(entry, span, verbose=False):
    if not span:
        return False

    response = entry["llm_response"]

    prev = ""
    delta = None
    for s in entry["sentence_data"]:
        if s.get("target") == "hallucinated":
            cur = s["cum_sentence"]
            delta = cur[len(prev):].strip()
            break
        prev = s["cum_sentence"]

    if not delta:
        return False

    extracted = response[span["start"]:span["end"]]

    if extracted == delta:
        return True

    if extracted.strip() == delta.strip():
        return True

    sim = SequenceMatcher(None, extracted, delta).ratio()
    return sim >= 0.9

In [ ]:
# =======================
# MAIN
# =======================

def main():
    with open(INPUT_JSON, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("原始題數:", len(data))

    minimal = []

    span_total = 0
    span_fail = 0

    for entry in tqdm(data, desc="Processing HalluRAG entries"):
        response = entry.get("llm_response")
        if not response:
            continue

        prompt_text = build_prompt_user_only(entry, debug=DEBUG)
        if not prompt_text:
            continue

        span = reverse_hallucination_span(entry)

        if VERIFY_SPAN and span:
            span_total += 1
            if not verify_hallucination_span(entry, span, verbose=DEBUG):
                span_fail += 1

        minimal.append({
            "prompt_text": prompt_text,
            "response": response,
            "hallucination_label": get_label(entry),
            "hallucination_span": span
        })

    print("Minimal 題數:", len(minimal))

    if VERIFY_SPAN and span_total > 0:
        acc = (span_total - span_fail) / span_total
        print(f"Span verification accuracy: {acc:.4f} ({span_total-span_fail}/{span_total})")

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(minimal, f, ensure_ascii=False, indent=2)

    print("Saved:", OUTPUT_JSON)

In [ ]:
if __name__ == "__main__":

    KEY = ["test", "train", "val"]
    for k in KEY:
        INPUT_JSON = f"/kaggle/input/hallurag-dataset/{k}_meta-llama_Llama-2-7b-chat-hf.json" # 拿取從 pickle 解開的檔案
        OUTPUT_JSON = f"hallurag_minimal_{k}.json"
        main()

# 合併三個檔案 (變為我要的一個檔案)

In [ ]:
import json

# =========================
# INPUT FILES
# =========================

FILES = {
    "/kaggle/working/hallurag_minimal_train.json": "train",
    "/kaggle/working/hallurag_minimal_val.json": "train", 
    "/kaggle/working/hallurag_minimal_test.json": "test",
}

OUTPUT_JSON = "hallurag_minimal_all.json"

In [ ]:
# =========================
# MERGE
# =========================

merged = []

for filename, split in FILES.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Loaded {filename}: {len(data)} samples → split={split}")

    for entry in data:
        entry = dict(entry)
        entry["split"] = split
        merged.append(entry)

In [ ]:
# =========================
# SAVE
# =========================

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print("Saved merged dataset:", OUTPUT_JSON)
print("Total samples:", len(merged))

In [ ]:
# =========================
# OPTIONAL CHECK
# =========================

from collections import Counter
cnt = Counter(e["split"] for e in merged)
print("Split distribution:", dict(cnt))

#  將資料變成Ragtruthful的分布 (幻覺資料佔比32%左右)

In [ ]:
import json
import random
from collections import defaultdict

TARGET_RATIO = 0.32
INPUT_JSON = "hallurag_minimal_all.json"
OUTPUT_JSON = "hallurag_minimal_all_balanced_32.json"

import random
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# ======================
# LOAD
# ======================
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Original total:", len(data))

# ======================
# GROUP BY SPLIT
# ======================
by_split = defaultdict(list)
for d in data:
    by_split[d["split"]].append(d)

final_data = []

# ======================
# PROCESS EACH SPLIT
# ======================
for split, items in by_split.items():
    pos = [d for d in items if d["hallucination_label"] == 1]
    neg = [d for d in items if d["hallucination_label"] == 0]

    H = len(pos)
    if H == 0:
        print(f"[WARN] {split} has no hallucinations, skipping balancing")
        final_data.extend(items)
        continue

    target_total = int(H / TARGET_RATIO)
    target_neg = max(target_total - H, 0)

    print(
        f"[{split.upper()}] "
        f"H={H}, N0={len(neg)} → keep N0={target_neg} "
        f"(target ratio ≈ {H/(H+target_neg):.2%})"
    )

    if target_neg >= len(neg):
        kept_neg = neg
    else:
        kept_neg = random.sample(neg, target_neg)

    final_data.extend(pos + kept_neg)

# ======================
# SHUFFLE & SAVE
# ======================
random.shuffle(final_data)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=2)

print("\nSaved balanced dataset:", OUTPUT_JSON)
print("Final total samples:", len(final_data))

# 觀看資料重複情況與分布

In [ ]:
import json
from collections import Counter, defaultdict

INPUT_JSON = "hallurag_minimal_all_balanced_32.json" 

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print("總樣本數:", len(data))

In [ ]:
# =========================
# 1) 檢查 prompt_text 重複
# =========================
prompt_counter = Counter(e["prompt_text"] for e in data)
dup_prompts = {p: c for p, c in prompt_counter.items() if c > 1}

print("\n=== prompt_text 重複情況 ===")
print("不同 prompt_text 數量:", len(prompt_counter))
print("有重複出現的 prompt_text 數量:", len(dup_prompts))

if dup_prompts:
    # 印前幾個範例
    print("\n出現次數最多的 5 個 prompt_text：")
    for p, c in sorted(dup_prompts.items(), key=lambda x: -x[1])[:5]:
        print(f"- 出現 {c} 次：{repr(p[:80])}...")

In [ ]:
# =========================
# 2) 檢查 response 重複
# =========================
resp_counter = Counter(e["response"] for e in data)
dup_resps = {r: c for r, c in resp_counter.items() if c > 1}

print("\n=== response 重複情況 ===")
print("不同 response 數量:", len(resp_counter))
print("有重複出現的 response 數量:", len(dup_resps))

if dup_resps:
    print("\n範例：出現次數最多的 5 個 response：")
    for r, c in sorted(dup_resps.items(), key=lambda x: -x[1])[:5]:
        print(f"- 出現 {c} 次：{repr(r[:80])}...")

In [ ]:
# =========================
# 3) 檢查 (prompt_text, response) 完整 pair 重複
# =========================
pair_counter = Counter((e["prompt_text"], e["response"]) for e in data)
dup_pairs = [pair for pair, c in pair_counter.items() if c > 1]

print("\n=== (prompt_text, response) pair 重複情況 ===")
print("不同 pair 數量:", len(pair_counter))
print("完全重複的 pair 數量:", len(dup_pairs))

if dup_pairs:
    print("\n 範例：前 3 個完全重複的 pair：")
    shown = 0
    for (p, r), c in pair_counter.items():
        if c > 1:
            print(f"\n--- 重複 {c} 次 ---")
            print("Prompt:", repr(p[:120]), "...")
            print("Response:", repr(r[:120]), "...")
            shown += 1
            if shown >= 3:
                break


In [ ]:
import json
import numpy as np
from collections import defaultdict

# ======================
# CONFIG
# ======================
INPUT_JSON = "hallurag_minimal_all_balanced_32.json"

# ======================
# LOAD
# ======================
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total samples:", len(data))

# ======================
# STAT CONTAINER
# ======================
stats = defaultdict(lambda: {
    "count": 0,
    "prompt_len": [],
    "resp_len": [],
    "has_hallucination": 0,
    "has_span": 0
})

# ======================
# COLLECT STATS
# ======================
for d in data:
    split = d.get("split", "unknown")

    prompt = d.get("prompt_text", "")
    resp = d.get("response", "")
    label = d.get("hallucination_label", 0)
    span = d.get("hallucination_span")

    bucket = stats[split]

    bucket["count"] += 1
    bucket["prompt_len"].append(len(prompt))
    bucket["resp_len"].append(len(resp))

    if label == 1:
        bucket["has_hallucination"] += 1

    if span is not None:
        bucket["has_span"] += 1


# ======================
# PRINT
# ======================
def print_stat(split, st):
    print(f"\n========== SPLIT: {split.upper()} ==========")
    print(f"Count: {st['count']}")
    print(f"Hallucinated (label=1): {st['has_hallucination']} "
          f"({st['has_hallucination']/st['count']:.2%})")
    print(f"With hallucination span: {st['has_span']} "
          f"({st['has_span']/st['count']:.2%})")

    print(
        f"Prompt length: max={max(st['prompt_len'])}, "
        f"avg={np.mean(st['prompt_len']):.1f}, "
        f"p95={np.percentile(st['prompt_len'], 95):.0f}"
    )
    print(
        f"Response length: max={max(st['resp_len'])}, "
        f"avg={np.mean(st['resp_len']):.1f}, "
        f"p95={np.percentile(st['resp_len'], 95):.0f}"
    )


for split in sorted(stats.keys()):
    print_stat(split, stats[split])


# 做與 RAGTruth 一樣的 Chunk 前處理

In [ ]:
# ============================================
# Chunk HalluRAG Minimal Dataset
# ============================================

import json
import nltk
from tqdm import tqdm

nltk.download("punkt")

In [ ]:
# ======================
# CONFIG
# ======================

INPUT_JSON = "hallurag_minimal_all_balanced_32.json"
OUTPUT_JSON = "hallurag_chunked_all_balanced_32.json"

PROMPT_CHUNK_SIZE = 500
RESPONSE_CHUNK_SIZE = 350
PROMPT_OVERLAP = 120
RESPONSE_OVERLAP = 80

SHOW_EXAMPLES = 2 

In [ ]:
# ======================
# Chunk Functions
# ======================

def smart_chunk_v2(text, chunk_size=350, overlap=120):
    """
    Char-based chunking, safe boundary
    Suitable for response
    """
    spans = []
    n = len(text)
    start = 0
    safe_chars = {'.', ',', ';', '!', '?', ' ', '\n'}

    while start < n:
        end = min(start + chunk_size, n)
        best = end

        while best > start + 20 and best < n and text[best - 1] not in safe_chars:
            best -= 1

        if best <= start + 20:
            best = end

        spans.append([start, best])

        next_start = best - overlap
        while 0 < next_start < n and text[next_start] not in safe_chars:
            next_start += 1

        if next_start <= start:
            next_start = best

        start = next_start

    return spans


def qa_hybrid_chunks(text, max_chunk_size=500, overlap=120):
    """
    Sentence-based + merge
    Suitable for prompt / RAG context
    """
    sentences = nltk.sent_tokenize(text)
    idx = 0

    sent_spans = []
    for s in sentences:
        start = text.index(s, idx)
        end = start + len(s)
        sent_spans.append((start, end))
        idx = end

    spans = []
    cur_start = None
    cur_end = None
    cur_len = 0
    safe_chars = {'.', ',', ';', '!', '?', ' ', '\n'}

    for start, end in sent_spans:
        if cur_start is None:
            cur_start = start
            cur_end = end
            cur_len = end - start
            continue

        if cur_len + (end - start) <= max_chunk_size:
            cur_end = end
            cur_len = cur_end - cur_start
        else:
            adj_end = cur_end
            while adj_end > cur_start + 20 and text[adj_end - 1] not in safe_chars:
                adj_end -= 1

            spans.append([cur_start, adj_end])

            next_start = max(adj_end - overlap, 0)
            while next_start < len(text) and text[next_start] not in safe_chars:
                next_start += 1

            cur_start = next_start
            cur_end = end
            cur_len = cur_end - cur_start

    if cur_start is not None:
        spans.append([cur_start, cur_end])

    return spans

In [ ]:
# ======================
# MAIN
# ======================

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

example_buffer = []

for entry in tqdm(data, desc="Adding span info"):
    prompt = entry["prompt_text"]
    response = entry["response"]

    entry["prompt_spans"] = qa_hybrid_chunks(
        prompt,
        max_chunk_size=PROMPT_CHUNK_SIZE,
        overlap=PROMPT_OVERLAP
    )

    entry["response_spans"] = qa_hybrid_chunks(
        response,
        max_chunk_size=RESPONSE_CHUNK_SIZE,
        overlap=RESPONSE_OVERLAP
    )

    if len(example_buffer) < SHOW_EXAMPLES:
        example_buffer.append(entry)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_JSON)


In [ ]:
print("\n==============================")
print("  Span Chunking Examples")
print("==============================")

for i, e in enumerate(example_buffer):
    print(f"\n========== SAMPLE {i+1} ==========")

    print("\n--- PROMPT CHUNKS ---")
    for j, (s, e_) in enumerate(e["prompt_spans"]):
        print(f"\n[Prompt Chunk {j}] ({s} ~ {e_})")
        print(e["prompt_text"][s:e_])

    print("\n--- RESPONSE CHUNKS ---")
    for j, (s, e_) in enumerate(e["response_spans"]):
        print(f"\n[Response Chunk {j}] ({s} ~ {e_})")
        print(e["response"][s:e_])